In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Trackingv
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")


python-dotenv could not parse statement starting at line 2


In [4]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")
print(llm)

c:\poc\GenAI\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001B8FC4C67B0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B8FC4C7230> root_client=<openai.OpenAI object at 0x000001B8FBF19D30> root_async_client=<openai.AsyncOpenAI object at 0x000001B8FC4C6F90> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********') stream_usa

In [5]:
## Input and get response form LLM

result=llm.invoke("What is generative AI?")

In [6]:
print(result)

content="Generative AI refers to a category of artificial intelligence systems designed to generate new content, such as text, images, audio, or code, by learning from existing data. Unlike traditional AI that typically analyzes and classifies data, generative AI models create new data that mimics the patterns and structure of the input data they were trained on. These models can generate original content that closely resembles what a human might produce.\n\nKey examples of generative AI include:\n\n1. **Text Generation**: Models like OpenAI's GPT (Generative Pre-trained Transformer) can produce written content that ranges from articles and poems to code and dialogue.\n\n2. **Image Generation**: Tools such as DALL-E and Stable Diffusion generate images from textual descriptions, allowing the creation of novel visual content.\n\n3. **Music and Audio Generation**: AI models can compose music or generate realistic-sounding speech and audio effects.\n\n4. **Video Generation**: Emerging tec

In [7]:
## Arxiv--Research
## Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [8]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [9]:

api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)


arxiv


In [10]:
tools=[wiki,arxiv]

In [13]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,OpenAIEmbeddings())


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B8AFC52120>, search_kwargs={})

In [30]:
class RetrieverTool:
    def __init__(self, retriever, name, description):
        self.retriever = retriever
        self.name = name
        self.description = description

    def __call__(self, query):
        return self.retriever.invoke(query)

# Usage
retriever_tool = RetrieverTool(retriever, "langsmith-search", "Search any information about Langsmith")
print(retriever_tool.name)
# To use:
results = retriever_tool("What is Langsmith?")
for doc in results:
    print(doc.page_content)

langsmith-search
LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.
It helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack, whether you’re using an existing framework or

In [31]:
tools=[wiki,arxiv,retriever_tool]

In [38]:
## Agents (Alternative Solution)
# If you do not have langchain_core.executors, use AgentExecutor from langchain.agents.agent_executor
from langchain.agents.agent_executor import AgentExecutor
from langchain.agents import create_tool_calling_agent

# 'prompt' should be a valid prompt template for your agent
# 'tools' should be a list of tool objects (as you have defined above)
# 'llm' is your language model (already defined)

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

response = agent_executor.invoke({"input": "What is Langsmith?"})
print(response)

ModuleNotFoundError: No module named 'langchain.agents.agent_executor'

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import create_retriever_tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

# 1. Setup the LLM (OpenAI)
# Use 'gpt-4o' or 'gpt-4o-mini' for the best tool-calling performance
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 2. Setup your Tools (Including your Retriever Tool)
# Assume 'retriever' was defined in your previous step
retriever_tool = create_retriever_tool(
    retriever, 
    "langsmith_search", 
    "Search for information about LangSmith. Use this for any LangSmith questions."
)

tools = [retriever_tool]

# 3. Create the Agent
# In 2026, 'create_react_agent' is the standard replacement for AgentExecutor
agent_executor = create_react_agent(llm, tools)

# 4. Invoke the Agent
# Input is passed as a list of messages
user_input = "What is LangSmith and how does it help with tracing?"
response = agent_executor.invoke({"messages": [("user", user_input)]})

# 5. Get the final answer
# The last message in the list is the agent's response
print(response["messages"][-1].content)